# Equity Valuation & Quant Risk Lab

Notebook guiado para análise de mercado, valuation simplificado e simulação de Monte Carlo. Projeto educacional; não constitui recomendação de investimento.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data import get_price_history, get_company_snapshot, get_cashflow_statement, extract_latest_free_cash_flow
from src.metrics import compute_market_metrics, log_returns, drawdown_series
from src.monte_carlo import simulate_gbm, summarize_simulation
from src.valuation import dcf_equity_value, dcf_sensitivity_table


## 1. Configuração

In [ ]:
TICKER = 'AAPL'
BENCHMARK = '^GSPC'
PERIOD = '5y'
RISK_FREE_RATE = 0.05

FORECAST_GROWTH = 0.06
WACC = 0.09
TERMINAL_GROWTH = 0.025
FORECAST_YEARS = 5

MC_DAYS = 252
MC_SIMS = 10_000
SEED = 42


## 2. Dados de mercado

In [ ]:
prices = get_price_history(TICKER, PERIOD)
benchmark_prices = get_price_history(BENCHMARK, PERIOD)
snapshot = get_company_snapshot(TICKER)
prices.tail(), snapshot


## 3. Preços e retornos

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(prices.index, prices.values)
ax.set_title(f'{TICKER} — Adjusted price history')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.grid(alpha=0.25)
plt.show()

returns = log_returns(prices)
returns.describe()


## 4. Métricas quantitativas

In [ ]:
metrics = compute_market_metrics(prices, benchmark_prices, RISK_FREE_RATE)
pd.Series(metrics, name='value')


In [ ]:
dd = drawdown_series(prices)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dd.index, dd.values)
ax.set_title(f'{TICKER} — Drawdown')
ax.set_ylabel('Drawdown')
ax.grid(alpha=0.25)
plt.show()


## 5. DCF simplificado

Os campos fundamentais são obtidos da fonte quando disponíveis. Caso algum valor esteja ausente, substitua por dados validados antes de interpretar o resultado.

In [ ]:
cashflow = get_cashflow_statement(TICKER)
fcf = extract_latest_free_cash_flow(cashflow)
cash = snapshot.get('totalCash')
debt = snapshot.get('totalDebt')
shares = snapshot.get('sharesOutstanding')

dcf_inputs = {
    'free_cash_flow': float(fcf),
    'cash': float(cash),
    'debt': float(debt),
    'shares_outstanding': float(shares),
}
dcf_inputs


In [ ]:
dcf = dcf_equity_value(
    **dcf_inputs,
    forecast_growth=FORECAST_GROWTH,
    wacc=WACC,
    terminal_growth=TERMINAL_GROWTH,
    forecast_years=FORECAST_YEARS,
)
pd.Series(dcf)


In [ ]:
sensitivity = dcf_sensitivity_table(
    **dcf_inputs,
    base_growth=FORECAST_GROWTH,
    base_wacc=WACC,
    terminal_growth=TERMINAL_GROWTH,
    forecast_years=FORECAST_YEARS,
)
sensitivity


## 6. Monte Carlo

In [ ]:
paths = simulate_gbm(prices, days=MC_DAYS, simulations=MC_SIMS, seed=SEED)
mc_summary = summarize_simulation(paths)
pd.Series(mc_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(paths[:, :100], alpha=0.2)
ax.set_title(f'{TICKER} — first 100 Monte Carlo paths')
ax.set_xlabel('Trading day')
ax.set_ylabel('Simulated price')
plt.show()

terminal = paths[-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(terminal, bins=60)
ax.axvline(pd.Series(terminal).quantile(0.05), linestyle='--', label='5th percentile')
ax.axvline(pd.Series(terminal).median(), linestyle='--', label='Median')
ax.set_title('Terminal price distribution')
ax.legend()
plt.show()


## 7. Como interpretar

- Compare retorno e volatilidade com o benchmark.
- Analise beta e drawdown, não apenas retorno médio.
- Trate o DCF como faixa sensível a premissas, não como preço exato.
- Compare cenários na matriz WACC × crescimento.
- Use Monte Carlo como exploração de cenários, não como previsão garantida.
